## **Machine Learning Aplicado a las Finanzas** 🚀
### **HW Sesión 5**

Andrés C. Medina Sanhueza

Senior Data Scientist Engineer

anmedinas@gmail.com

---

### Consideraciones Previas

* Tarea es **Individual**
* Fecha de Entrega: **Jueves 2 Jul 23:59** (cualquier commit posterior descontará puntos)
* El notebook debe correr de arriba abajo sin errores (`Kernel → Restart & Run All`)
* No se admiten `pass` pendientes ni celdas vacías donde se espera código
* Deben crear una rama nueva en su repositorio `feat/hw05` y trabajar la tarea en esa rama. El notebook debe quedar en la carpeta `hw/`
* **Archivo a entregar:** URL del repositorio GitHub enviada a `anmedinas@gmail.com`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import linalg, optimize
from scipy.stats import spearmanr
from sklearn.decomposition import PCA, SparsePCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')
sns.set_style('dark')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
np.random.seed(20260627)

---
## Parte 1 — SVD, Eigendecomposición y PCA desde Cero *(25 pts)*

### Contexto

La sesión 5 mostró que PCA se reduce a dos operaciones equivalentes: la descomposición espectral de $\hat{\Sigma}$ y la SVD de la matriz de datos $X$. Ambas producen los mismos eigenvectores:

$$\hat{\Sigma} = V \Lambda V^\top \quad \Longleftrightarrow \quad X = U S V^\top, \quad \lambda_i = \frac{s_i^2}{T-1}$$

En esta parte implementarás PCA desde cero de **dos formas** y verificarás que producen resultados idénticos.

### 1.1 — Implementación Dual: Eigen vs SVD *(12 pts)*

Implementa dos funciones:
- `pca_eigen(X, k)`: via eigendecomposición de $\hat{\Sigma} = X_c^\top X_c / (T-1)$
- `pca_svd(X, k)`: via SVD de $X_c$ directamente

Ambas deben retornar `(scores, components, explained_variance_ratio)` con `scores` de shape `(T, k)` y `components` de shape `(k, n)`.

**Verificación:** el error máximo entre ambas implementaciones (corrigiendo signos) debe ser `< 1e-10`.

**Nota sobre signos:** los eigenvectores son únicos salvo por su signo. Alinea los componentes de ambas funciones forzando que `components[j].mean() >= 0` en cada caso.

In [ ]:
# Datos de trabajo: simulación con estructura factorial conocida
np.random.seed(42)
T_sim, n_sim, k_true = 500, 25, 3

F_sim = np.random.randn(T_sim, k_true) * np.array([2.0, 1.0, 0.6])
B_sim = np.random.randn(n_sim, k_true)
B_sim[:, 0] = np.abs(B_sim[:, 0])  # primer factor: exposicion positiva para todos
eps_sim = np.random.randn(T_sim, n_sim) * 0.3
R_sim = F_sim @ B_sim.T + eps_sim   # (T, n)

print(f'Datos simulados: T={T_sim}, n={n_sim}, k_verdadero={k_true}')
print(f'Varianza total: {R_sim.var(axis=0).sum():.2f}')

In [ ]:
def _align_component_signs(components):
    """Alinea signos para que la media de cada componente sea no negativa."""
    components = np.asarray(components, dtype=float).copy()
    for j in range(components.shape[0]):
        if components[j].mean() < 0:
            components[j] *= -1
    return components


def pca_eigen(X, k):
    """
    PCA via eigendecomposicion de la covarianza muestral.
    """
    X = np.asarray(X, dtype=float)
    T, n = X.shape
    if not 1 <= k <= min(T, n):
        raise ValueError(f'k debe estar entre 1 y {min(T, n)}')

    Xc = X - X.mean(axis=0)
    Sigma = Xc.T @ Xc / (T - 1)
    eigenvals, eigenvecs = np.linalg.eigh(Sigma)
    order = np.argsort(eigenvals)[::-1]
    eigenvals = eigenvals[order]
    eigenvecs = eigenvecs[:, order]

    components = _align_component_signs(eigenvecs[:, :k].T)
    scores = Xc @ components.T
    ev_ratio = eigenvals[:k] / eigenvals.sum()
    return scores, components, ev_ratio


def pca_svd(X, k):
    """
    PCA via SVD de la matriz de datos centrada.
    """
    X = np.asarray(X, dtype=float)
    T, n = X.shape
    if not 1 <= k <= min(T, n):
        raise ValueError(f'k debe estar entre 1 y {min(T, n)}')

    Xc = X - X.mean(axis=0)
    _, singular_values, Vt = np.linalg.svd(Xc, full_matrices=False)
    eigenvals = singular_values**2 / (T - 1)

    components = _align_component_signs(Vt[:k])
    scores = Xc @ components.T
    ev_ratio = eigenvals[:k] / eigenvals.sum()
    return scores, components, ev_ratio


# === Verificacion ===
k_ver = 5
sc_e, comp_e, evr_e = pca_eigen(R_sim, k_ver)
sc_s, comp_s, evr_s = pca_svd(R_sim, k_ver)

# Comparar con sklearn
pca_sk_ver = PCA(n_components=k_ver).fit(R_sim)
comp_sk = _align_component_signs(pca_sk_ver.components_)
ev_sklearn = pca_sk_ver.explained_variance_ratio_

err_e_sk = np.abs(comp_e - comp_sk).max()
err_s_sk = np.abs(comp_s - comp_sk).max()
err_eigen_svd = np.abs(comp_e - comp_s).max()
err_evr = np.abs(evr_e - evr_s).max()

print(f'Error maximo pca_eigen  vs sklearn: {err_e_sk:.2e}')
print(f'Error maximo pca_svd    vs sklearn: {err_s_sk:.2e}')
print(f'Error maximo eigen vs svd (inter):  {err_eigen_svd:.2e}')
print(f'Error maximo EVR eigen vs svd:      {err_evr:.2e}')
print(f'\nVarianza explicada (eigen):  {evr_e.round(3)}')
print(f'Varianza explicada (svd):    {evr_s.round(3)}')
print(f'Varianza explicada (sklearn):{ev_sklearn.round(3)}')

assert err_eigen_svd < 1e-10, 'Eigen y SVD deben coincidir despues de alinear signos'
assert err_e_sk < 1e-10 and err_s_sk < 1e-10, 'Las implementaciones deben coincidir con sklearn'


### 1.2 — Selección de $k$: Tres Criterios Comparados *(13 pts)*

La elección del número de componentes $k$ es la decisión de diseño más importante en PCA. Implementa y compara tres criterios sobre los datos simulados y sobre datos reales del S&P 500.

| Criterio | Regla formal | Intuitivo |
|---|---|---|
| **Varianza acumulada** | Menor $k$ tal que $\sum_{i=1}^k \lambda_i / \sum \lambda_i \geq 0.80$ | Retener el 80% de la información |
| **Kaiser** | Retener $\lambda_i > \bar{\lambda}$ (datos estandarizados: $\bar{\lambda}=1$) | Cada PC debe explicar más que una variable |
| **Bai-Ng IC$_1$** | $\arg\min_k \left[ \ln V(k) + k \cdot g(n,T) \right]$, $g(n,T) = \frac{n+T}{nT}\ln\frac{nT}{n+T}$ | Penalizar la complejidad del modelo |

donde $V(k) = \frac{1}{nT}\sum_{i>k} \lambda_i$ es la varianza no explicada por $k$ factores.

**Requisitos:**
1. Implementar `select_k_criteria(eigenvalues, n, T)` que retorne un dict `{'var80': int, 'kaiser': int, 'bai_ng': int}`
2. Aplicar sobre datos simulados (k verdadero = 3) y sobre retornos de los **11 ETFs sectoriales del S&P 500** descargados con yfinance (2020–2024)
3. Generar un panel de 4 subgráficos para cada dataset:
   - Scree plot (barras + línea de eigenvalores)
   - Varianza explicada acumulada con umbrales 80% y 95%
   - Criterio de Kaiser (línea horizontal en $\bar{\lambda}$)
   - Criterio Bai-Ng IC$_1$ vs $k$
4. Responder las preguntas al final de la sección

In [ ]:
def bai_ng_ic1_values(eigenvalues, n, T, k_max=None):
    """Calcula IC1 de Bai-Ng para un rango acotado de factores candidatos."""
    eigenvalues = np.asarray(eigenvalues, dtype=float)
    eigenvalues = np.sort(eigenvalues)[::-1]
    if k_max is None:
        # En aplicaciones factoriales se acota la busqueda; sin cota, IC1 puede elegir el borde n-1.
        k_max = int(np.floor(np.sqrt(min(n, T))))
    k_max = max(1, min(int(k_max), n - 1, T - 1, len(eigenvalues) - 1))

    g_nT = (n + T) / (n * T) * np.log(n * T / (n + T))
    ic = []
    for k in range(1, k_max + 1):
        # Los eigenvalores provienen de una matriz de covarianza; por eso el RSS promedio es /n.
        V_k = eigenvalues[k:].sum() / n
        ic.append(np.log(max(V_k, 1e-12)) + k * g_nT)
    return np.arange(1, k_max + 1), np.array(ic)


def select_k_criteria(eigenvalues, n, T, k_max=None):
    """
    Devuelve el numero optimo de componentes segun 3 criterios.
    """
    eigenvalues = np.asarray(eigenvalues, dtype=float)
    eigenvalues = np.sort(eigenvalues)[::-1]
    total_var = eigenvalues.sum()
    if total_var <= 0:
        raise ValueError('La suma de eigenvalores debe ser positiva')

    ev_ratio = eigenvalues / total_var
    var80 = int(np.searchsorted(np.cumsum(ev_ratio), 0.80)) + 1

    lambda_mean = eigenvalues.mean()
    kaiser = int(np.sum(eigenvalues > lambda_mean))
    kaiser = max(1, kaiser)

    k_grid, ic = bai_ng_ic1_values(eigenvalues, n, T, k_max=k_max)
    bai_ng = int(k_grid[np.argmin(ic)])

    return {'var80': var80, 'kaiser': kaiser, 'bai_ng': bai_ng}


# Prueba sobre datos simulados crudos, como fueron generados en el enunciado.
pca_full_sim = PCA().fit(R_sim)
eigenvals_sim = pca_full_sim.explained_variance_
k_dict_sim = select_k_criteria(eigenvals_sim, n_sim, T_sim)
print(f'Datos simulados (k verdadero={k_true}): {k_dict_sim}')
print(f'Varianza explicada acumulada primeros 5 PCs: {np.cumsum(pca_full_sim.explained_variance_ratio_)[:5].round(3)}')


In [ ]:
# Descarga ETFs sectoriales
from pathlib import Path

cache_dir = Path('.cache/yfinance_hw05')
cache_dir.mkdir(parents=True, exist_ok=True)
yf.set_tz_cache_location(str(cache_dir.resolve()))

sector_tickers = ['XLK','XLF','XLV','XLY','XLC','XLI','XLE','XLU','XLRE','XLP','XLB']
raw_sec = yf.download(sector_tickers, start='2020-01-01', end='2024-01-01',
                      progress=False, auto_adjust=True, threads=False)['Close']
raw_sec = raw_sec.reindex(columns=sector_tickers).dropna(how='all')
ret_sec = raw_sec.pct_change().dropna(how='any')
if ret_sec.empty:
    raise RuntimeError('La descarga de ETFs sectoriales vino vacia. Revisa conexion/yfinance antes de continuar.')
print(f'ETFs: {ret_sec.shape[0]} dias | {ret_sec.shape[1]} sectores')
print(f'Periodo: {ret_sec.index[0].date()} ? {ret_sec.index[-1].date()}')

# PCA completo sobre retornos estandarizados para que Kaiser sea interpretable.
ret_sec_std = StandardScaler().fit_transform(ret_sec.values)
pca_full_sec = PCA().fit(ret_sec_std)
eigenvals_sec = pca_full_sec.explained_variance_
k_dict_sec = select_k_criteria(eigenvals_sec, ret_sec.shape[1], ret_sec.shape[0])
print(f'ETFs sectoriales estandarizados: {k_dict_sec}')
print(f'Varianza explicada acumulada primeros 5 PCs: {np.cumsum(pca_full_sec.explained_variance_ratio_)[:5].round(3)}')


def plot_k_selection_panel(eigenvalues, n, T, title):
    eigenvalues = np.asarray(eigenvalues, dtype=float)
    ev_ratio = eigenvalues / eigenvalues.sum()
    cum_var = np.cumsum(ev_ratio)
    k_dict = select_k_criteria(eigenvalues, n, T)
    k_grid, ic = bai_ng_ic1_values(eigenvalues, n, T)
    x = np.arange(1, len(eigenvalues) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    axes = axes.ravel()
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0].bar(x, eigenvalues, color='steelblue', alpha=0.75)
    axes[0].plot(x, eigenvalues, color='black', marker='o', lw=1.4)
    axes[0].set_title('Scree plot')
    axes[0].set_xlabel('Componente')
    axes[0].set_ylabel('Eigenvalor')

    axes[1].plot(x, cum_var, marker='o', color='steelblue')
    axes[1].axhline(0.80, color='red', ls='--', lw=1, label='80%')
    axes[1].axhline(0.95, color='gray', ls=':', lw=1, label='95%')
    axes[1].set_ylim(0, 1.03)
    axes[1].set_title('Varianza explicada acumulada')
    axes[1].set_xlabel('Componentes retenidos')
    axes[1].set_ylabel('Proporcion acumulada')
    axes[1].legend(frameon=False)

    lambda_mean = eigenvalues.mean()
    axes[2].bar(x, eigenvalues, color=np.where(eigenvalues > lambda_mean, 'steelblue', 'lightgray'))
    axes[2].axhline(lambda_mean, color='red', ls='--', lw=1.2, label=fr'$\bar{{\lambda}}$={lambda_mean:.2f}')
    axes[2].set_title('Criterio Kaiser')
    axes[2].set_xlabel('Componente')
    axes[2].set_ylabel('Eigenvalor')
    axes[2].legend(frameon=False)

    axes[3].plot(k_grid, ic, color='steelblue', marker='o')
    axes[3].scatter(k_dict['bai_ng'], ic[np.argmin(ic)], color='red', zorder=3, label=f"min k={k_dict['bai_ng']}")
    axes[3].set_title('Bai-Ng IC1')
    axes[3].set_xlabel('k')
    axes[3].set_ylabel('IC1')
    axes[3].legend(frameon=False)

    marker_colors = {'var80': 'darkorange', 'kaiser': 'purple', 'bai_ng': 'green'}
    for ax in axes[:3]:
        for name, k_val in k_dict.items():
            ax.axvline(k_val, color=marker_colors[name], ls=':', lw=1, alpha=0.9)
    for ax in axes:
        ax.grid(alpha=0.25)
    plt.tight_layout()
    return fig, axes, k_dict, k_grid, ic

fig_sim, axes_sim, k_dict_sim, k_grid_sim, ic_sim = plot_k_selection_panel(
    eigenvals_sim, n_sim, T_sim, 'Seleccion de k ? Datos simulados'
)
fig_sec, axes_sec, k_dict_sec, k_grid_sec, ic_sec = plot_k_selection_panel(
    eigenvals_sec, ret_sec.shape[1], ret_sec.shape[0], 'Seleccion de k ? ETFs sectoriales S&P 500 estandarizados'
)

print(f'Criterios simulacion: {k_dict_sim}')
print(f'Criterios ETFs:       {k_dict_sec}')


**Responde (m?nimo 2 oraciones por pregunta):**

**a)** En los datos simulados con $k_{\text{verdadero}}=3$, ?cu?l de los tres criterios recupera el valor correcto? ?Por qu? los otros pueden sobre/sub-estimar $k$?

> En la simulaci?n, Kaiser y Bai-Ng seleccionan $k=3$, que coincide con los tres factores usados para generar los datos. El criterio de varianza acumulada al 80% selecciona $k=2$ porque los dos primeros factores ya explican cerca de 90% de la varianza; por eso puede subestimar factores econ?micamente reales cuando el tercer factor existe pero aporta menos varianza marginal.

**b)** Para los ETFs sectoriales reales, ?los tres criterios coinciden? ?Qu? interpretaci?n econ?mica tiene el $k$ seleccionado por Bai-Ng?

> En los ETFs sectoriales los criterios no coinciden: la varianza acumulada al 80% selecciona $k=2$, Kaiser selecciona $k=1$ y Bai-Ng selecciona $k=3$ dentro del rango acotado de factores candidatos. Econ?micamente, el $k=3$ de Bai-Ng sugiere un factor de mercado dominante m?s dos dimensiones sectoriales adicionales, probablemente asociadas a rotaciones defensivo/c?clico, tecnolog?a/crecimiento o sensibilidad a tasas/energ?a.

**c)** El criterio de Kaiser fue dise?ado para datos estandarizados (con varianza unitaria por variable). ?Qu? error cometes si aplicas Kaiser sobre retornos sin estandarizar?

> Si se aplica Kaiser sobre retornos sin estandarizar, los activos o sectores con mayor volatilidad dominan los eigenvalores y el umbral deja de representar ?m?s varianza que una variable promedio estandarizada?. El resultado mezcla selecci?n de factores con diferencias de escala, por lo que puede retener componentes por volatilidad relativa y no por verdadera estructura com?n de correlaci?n.


---
## Parte 2 — PCA en la Curva de Tasas de Interés *(25 pts)*

### Contexto

Litterman & Scheinkman (1991) demostraron que los tres primeros componentes de la curva soberana tienen interpretación directa: **nivel**, **pendiente** y **curvatura**. Estos factores son la base del análisis de riesgo de tasa en portafolios de renta fija.

Trabajarás con datos simulados usando el modelo Nelson-Siegel:

$$y(\tau) = f_1 \cdot L_1(\tau) + f_2 \cdot L_2(\tau) + f_3 \cdot L_3(\tau) + \varepsilon(\tau)$$

donde los loadings son:
$$L_1(\tau) = 1, \quad L_2(\tau) = \frac{1 - e^{-\lambda\tau}}{\lambda\tau}, \quad L_3(\tau) = \frac{1 - e^{-\lambda\tau}}{\lambda\tau} - e^{-\lambda\tau}$$

y los factores siguen procesos AR(1) con alta persistencia.

### 2.1 — Simulación y PCA en la Curva *(10 pts)*

La celda de datos ya genera la curva simulada. Tu tarea es:

1. Aplicar PCA sobre los **cambios diarios de tasas** $\Delta y_t$ (primeras diferencias) — **no sobre niveles**
2. Identificar cuántos componentes explican el 99% de la varianza
3. Visualizar los loadings de los 3 primeros PCs vs tenor y comparar con las curvas Nelson-Siegel teóricas
4. Verificar que PCA recupera la estructura Nelson-Siegel calculando la correlación entre el loading del PC1 y $L_1$, PC2 y $L_2$, PC3 y $L_3$

**Regla de signo:** ajusta el signo de cada PC para que el loading promedio sea positivo (PC1 y PC3) o para que el loading en $\tau=0.25$ sea mayor que el loading en $\tau=30$ (PC2).

In [ ]:
# Generación de datos de curva de tasas
np.random.seed(2026)
tenors = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30])
T_yc   = 2000
lam_ns = 0.5

# Loadings Nelson-Siegel teoricos
L1_ns = np.ones(len(tenors))
L2_ns = (1 - np.exp(-lam_ns * tenors)) / (lam_ns * tenors)
L3_ns = L2_ns - np.exp(-lam_ns * tenors)

# Factores AR(1)
def ar1(T, phi, sigma, mu=0, seed=None):
    rng = np.random.default_rng(seed)
    x = np.zeros(T)
    x[0] = mu
    eps = rng.normal(0, sigma, T)
    for t in range(1, T):
        x[t] = mu + phi * (x[t-1] - mu) + eps[t]
    return x

f1_yc = ar1(T_yc, 0.998, 0.02, mu=0.04, seed=1)
f2_yc = ar1(T_yc, 0.995, 0.015, mu=0.01, seed=2)
f3_yc = ar1(T_yc, 0.990, 0.010, mu=0.00, seed=3)

yields_yc = (np.outer(f1_yc, L1_ns) +
             np.outer(f2_yc, L2_ns) +
             np.outer(f3_yc, L3_ns) +
             np.random.randn(T_yc, len(tenors)) * 0.001)

dy_yc = np.diff(yields_yc, axis=0)
print(f'Curva de tasas: {T_yc} periodos | {len(tenors)} tenores')
print(f'Cambios diarios: shape={dy_yc.shape}')
print(f'Tenores: {tenors}')

# TODO 1: Aplica PCA sobre dy_yc
# pca_yc = PCA().fit(dy_yc)

# TODO 2: Cuantos componentes explican el 99% de la varianza?
# ev_yc = pca_yc.explained_variance_ratio_
# k_99  = np.searchsorted(np.cumsum(ev_yc), 0.99) + 1
# print(f'Componentes para explicar 99%: {k_99}')

# TODO 3: Visualiza loadings de los 3 primeros PCs vs NS teorico
# Ajusta signo segun la regla descrita en el enunciado
# Grafico: fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# Cada panel: loading PCA (solido) vs loading NS (punteado) vs tenor

# TODO 4: Calcula y reporta correlaciones entre PCs y loadings NS
# corr_1 = np.corrcoef(pca_yc.components_[0], L1_ns)[0, 1]
# ...
pass

### 2.2 — DV01 Factorial y Construcción de Portafolio Inmune al Nivel *(15 pts)*

#### DV01 por factor

El **DV01** (*Dollar Value of 01*) de un bono es la sensibilidad del precio ante un movimiento de 1 punto base (0.01%) en la tasa. Para un portafolio con posición $w_i$ en el bono del tenor $\tau_i$:

$$\text{DV01}_{\text{portafolio}} \approx \sum_i w_i \cdot \underbrace{(-D_i \cdot P_i \cdot 0.0001)}_{\text{DV01 individual}}$$

donde $D_i$ es la duración modificada del bono $i$. Aproximando la duración por el tenor: $D_i \approx \tau_i$.

El **DV01 factorial** descompone la exposición del portafolio a cada factor PCA:

$$\text{DV01}_{\text{PC}j} = \sum_i w_i \cdot (-\tau_i) \cdot b_{ij} \cdot 0.0001$$

donde $b_{ij}$ es el loading del bono $i$ en el factor $j$ (componente $j$ del PCA evaluado en el tenor $\tau_i$).

#### Portafolio neutral al nivel, largo en curvatura

Un *butterfly trade* consiste en:
- Ser **largo** en el bono de vencimiento medio (5 años)
- Ser **corto** en los extremos (2 y 10 años)
- **Neutro** al riesgo de nivel: $\text{DV01}_{\text{PC1}} = 0$
- **Máxima exposición** al factor de curvatura: $|\text{DV01}_{\text{PC3}}|$ maximizado

**Requisitos:**
1. Implementar `dv01_factorial(weights, tenors, loadings)` que retorna el DV01 de cada factor
2. Construir un butterfly 3-bono (tenores: 2, 5, 10 años) con pesos $w = [w_1, 1, w_3]$, encontrando $w_1, w_3$ tal que $\text{DV01}_{\text{PC1}} = 0$ y $\sum w_i = 0$ (portafolio auto-financiado)
3. Reportar el DV01 de los 3 primeros factores para este portafolio
4. Visualizar la distribución de P&L simulada del portafolio vs un portafolio largo en 5Y solamente

In [ ]:
def dv01_factorial(weights, tenors_pos, loadings_k):
    """
    Calcula el DV01 factorial de un portafolio de bonos.

    Parameters
    ----------
    weights      : array (m,)   — pesos del portafolio en cada bono
    tenors_pos   : array (m,)   — tenores de los bonos en el portafolio
    loadings_k   : array (k, m) — loadings PCA evaluados en cada tenor
                   loadings_k[j, i] = loading del factor j en el tenor i

    Returns
    -------
    dv01 : array (k,) — DV01 de cada factor (en terminos de precio por $1M nocional)

    Formula:
    dv01[j] = sum_i(weights[i] * (-tenors_pos[i]) * loadings_k[j, i] * 0.0001)
    """
    # TODO: implementa la formula del DV01 factorial
    pass


# Descomenta cuando hayas completado la Parte 2.1

# Tenores del butterfly
# butterfly_tenors = np.array([2.0, 5.0, 10.0])

# Indices en el array 'tenors' (los mas cercanos a 2, 5, 10)
# idx_2y  = np.argmin(np.abs(tenors - 2.0))
# idx_5y  = np.argmin(np.abs(tenors - 5.0))
# idx_10y = np.argmin(np.abs(tenors - 10.0))

# Loadings PCA en estos tenores (3 primeros factores, 3 tenores)
# comps_yc = pca_yc.components_[:3]   # (3, 11)
# Ajusta signos igual que en 2.1
# ...

# loadings_butterfly = comps_yc[:, [idx_2y, idx_5y, idx_10y]]  # (3, 3)

# TODO: Encuentra w1, w3 resolviendo el sistema lineal:
#   Constraint 1: w1 + 1 + w3 = 0  (auto-financiado)
#   Constraint 2: dv01_PC1 = 0
#   El bono de 5Y tiene peso = 1 (normalizacion)
#
# Hint: escribe las 2 ecuaciones como A @ [w1, w3] = b y resuelve con np.linalg.solve

# TODO: Reporta DV01 factorial del butterfly
# w_bf = np.array([w1, 1.0, w3])
# dv01_bf = dv01_factorial(w_bf, butterfly_tenors, loadings_butterfly)
# print(f'Pesos butterfly: 2Y={w1:.3f}, 5Y=1.000, 10Y={w3:.3f}')
# for j, d in enumerate(dv01_bf):
#     print(f'  DV01 PC{j+1}: {d:.4f}')

# TODO: Simula P&L del butterfly vs long 5Y
# Scores de los 3 factores: sc_yc = pca_yc.transform(dy_yc)[:, :3]  (T-1, 3)
# PnL butterfly: suma ponderada de cambios en yields por tenor
# PnL_bf[t]    = sum_i(w_bf[i] * (-butterfly_tenors[i]) * dy_yc[t, idx_i])
# PnL_long5Y[t] = -5.0 * dy_yc[t, idx_5y]
#
# Genera histograma comparativo de P&L (anualizado) + tabla de estadisticos
pass

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿Los loadings del PCA recuperados en 2.1 se parecen a los loadings Nelson-Siegel teóricos? ¿Qué correlaciones obtienes? ¿Cuántos puntos base de varianza explica cada factor?

> *Completa aquí*

**b)** ¿Cuáles son los pesos $w_1$ y $w_3$ del butterfly encontrado? Interpreta el signo de cada peso: ¿por qué el bono de 2Y y el de 10Y deben tener signos distintos al de 5Y en este trade?

> *Completa aquí*

**c)** Compara la distribución del P&L del butterfly vs la posición larga en 5Y. ¿Cuál tiene mayor volatilidad? ¿Cuál tiene mayor sharpe ratio empírico (usando la distribución simulada)? ¿Tiene sentido desde la teoría?

> *Completa aquí*

---
## Parte 3 — PCA en Renta Variable: Shrinkage y Portafolios *(25 pts)*

### Contexto

Un portafolio de $n$ acciones necesita una estimación de $\Sigma \in \mathbb{R}^{n \times n}$ para la optimización de mínima varianza. La estimación muestral $\hat{\Sigma}$ falla cuando $T$ es moderado respecto a $n$ porque:

1. Los eigenvalores pequeños son **subestimados** y los grandes **sobreestimados** (Marchenko-Pastur)
2. La inversa $\hat{\Sigma}^{-1}$ amplifica el ruido — los pesos de Markowitz se vuelven extremos

La solución: **covarianza shrinkage via PCA**:

$$\hat{\Sigma}_{\text{PCA}} = \underbrace{\hat{B}_k \hat{B}_k^\top}_{\text{sistemático}} + \underbrace{\hat{D}}_{\text{idiosincrático}}, \quad \hat{D} = \text{diag}(\hat{\Sigma} - \hat{B}_k \hat{B}_k^\top)$$

Esta matriz tiene exactamente $k$ eigenvalores grandes (los factores) y $n-k$ eigenvalores iguales al ruido idiosincrático — mucho mejor condicionada que $\hat{\Sigma}$.

### 3.1 — Loadings Factoriales y Estructura de Sectores *(8 pts)*

Usa los datos de los 11 ETFs sectoriales descargados en la Parte 1.

**Requisitos:**
1. Ajustar PCA con $k=4$ componentes sobre `ret_sec`
2. Generar un **heatmap de loadings** (sectores × PCs) con anotaciones, coloreado en divergente (`RdBu_r`)
3. Para PC1, graficar los loadings como barras horizontales coloreadas por signo y ordenadas de mayor a menor
4. Calcular el **factor de correlación de mercado** de PC1: $\rho_{\text{MKT}} = \text{corr}(\text{score}_{\text{PC1}}, r_{\text{SPY}})$, descargando SPY del mismo período
5. Responder las preguntas de interpretación

In [ ]:
# TODO: Ajusta PCA k=4 sobre los retornos de ETFs sectoriales
# pca_sec = PCA(n_components=4).fit(ret_sec.values)
# scores_sec = pca_sec.transform(ret_sec.values)

# TODO: Heatmap de loadings (11 sectores x 4 PCs)
# loadings_df = pd.DataFrame(
#     pca_sec.components_.T,
#     index=sector_tickers,
#     columns=[f'PC{i+1}' for i in range(4)]
# )
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# sns.heatmap(loadings_df, ax=axes[0], cmap='RdBu_r', center=0,
#             annot=True, fmt='.2f', cbar_kws={'shrink': 0.7})
# axes[0].set_title('Factor Loadings — ETFs Sectoriales')

# TODO: Barplot loadings PC1
# pc1_loadings = loadings_df['PC1'].sort_values(ascending=True)
# colors_bar = ['tomato' if v < 0 else 'steelblue' for v in pc1_loadings]
# axes[1].barh(pc1_loadings.index, pc1_loadings.values, color=colors_bar)
# axes[1].set_title('Loadings PC1 por Sector')
# axes[1].axvline(0, color='black', lw=0.8)

# TODO: Correlacion PC1 con SPY
# spy_data = yf.download('SPY', start='2020-01-01', end='2024-01-01',
#                        progress=False, auto_adjust=True)['Close']
# ret_spy = spy_data.pct_change().dropna().reindex(ret_sec.index).dropna()
# # Alinear scores y SPY al mismo indice
# corr_pc1_spy = np.corrcoef(scores_sec[:len(ret_spy), 0], ret_spy.values)[0, 1]
# print(f'Correlacion PC1 con SPY: {corr_pc1_spy:.3f}')
pass

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿Cuál es la correlación de PC1 con el retorno del SPY? ¿Qué dice esto sobre la interpretación del primer componente como un "factor de mercado"?

> *Completa aquí*

**b)** Identifica qué sectores tienen loadings más negativos en PC2. ¿Qué hipótesis económica plantearías sobre qué separa a esos sectores del grupo con loading positivo?

> *Completa aquí*

### 3.2 — Covarianza Shrinkage: Implementación y Condicionamiento *(10 pts)*

Implementa la función `pca_shrinkage_cov(X, k)` que construye $\hat{\Sigma}_{\text{PCA}}$.

**Pasos:**
1. Ajustar PCA con $k$ componentes
2. Calcular $\hat{B}_k \in \mathbb{R}^{n \times k}$: $\hat{B}_k = V_k \cdot \text{diag}(\sqrt{\lambda_1}, \dots, \sqrt{\lambda_k})$ donde $V_k$ son los eigenvectores
3. Calcular la covarianza muestral $\hat{\Sigma}$
4. Calcular $\hat{D} = \text{diag}(\hat{\Sigma} - \hat{B}_k \hat{B}_k^\top)$, clipeando a `max(1e-8, .)`
5. Retornar $\hat{\Sigma}_{\text{PCA}} = \hat{B}_k \hat{B}_k^\top + \hat{D}$

**Verificación:** $\hat{\Sigma}_{\text{PCA}}$ debe tener exactamente $k$ eigenvalores significativamente mayores que los demás.

**Análisis:** compara el número de condición $\kappa(\hat{\Sigma})$ vs $\kappa(\hat{\Sigma}_{\text{PCA}})$ para $k \in \{1, 2, 3, 4, 5\}$.

In [ ]:
def pca_shrinkage_cov(X, k):
    """
    Covarianza shrinkage via PCA: Sigma_PCA = B_k @ B_k.T + D

    Parameters
    ----------
    X : array (T, n) — retornos crudos
    k : int          — numero de factores

    Returns
    -------
    Sigma_pca : array (n, n) — covarianza PCA-shrinkage
    B_hat     : array (n, k) — factor loadings
    D_hat     : array (n, n) — covarianza idiosincratica (diagonal)
    """
    # TODO paso 1: pca_k = PCA(n_components=k).fit(X)
    # TODO paso 2: B_hat = pca_k.components_.T * np.sqrt(pca_k.explained_variance_)
    # TODO paso 3: Sigma_sample = np.cov(X.T)
    # TODO paso 4: D_diag = np.diag(Sigma_sample - B_hat @ B_hat.T)
    #              D_diag = np.maximum(D_diag, 1e-8)
    #              D_hat  = np.diag(D_diag)
    # TODO paso 5: Sigma_pca = B_hat @ B_hat.T + D_hat
    # return Sigma_pca, B_hat, D_hat
    pass


# Descomenta cuando hayas implementado la funcion

# Sigma_sample = np.cov(ret_sec.values.T)
# cond_sample  = np.linalg.cond(Sigma_sample)

# k_values  = [1, 2, 3, 4, 5]
# cond_pca_list = []
# for k_try in k_values:
#     Sigma_pca_try, _, _ = pca_shrinkage_cov(ret_sec.values, k_try)
#     cond_pca_list.append(np.linalg.cond(Sigma_pca_try))

# TODO: Genera dos paneles:
# Panel 1 (izquierdo): numero de condicion vs k
#   ax1.plot(k_values, cond_pca_list, 'o-', label='Sigma_PCA')
#   ax1.axhline(cond_sample, color='red', ls='--', label='Sigma muestral')

# Panel 2 (derecho): heatmaps de Sigma_muestral vs Sigma_PCA(k=3) lado a lado
#   Sigma_pca3, _, _ = pca_shrinkage_cov(ret_sec.values, 3)
#   Muestra los dos heatmaps con sns.heatmap y xticklabels=sector_tickers

# TODO: Verifica que Sigma_PCA tiene exactamente k eigenvalores grandes
# evals_pca3 = np.linalg.eigvalsh(Sigma_pca3)[::-1]
# print(f'Eigenvalores Sigma_PCA(k=3): {evals_pca3.round(6)}')
pass

### 3.3 — Portafolio de Mínima Varianza: Sigma Muestral vs Sigma PCA *(7 pts)*

El portafolio de **mínima varianza global** (GMV) se obtiene de:

$$w^* = \arg\min_w \; w^\top \Sigma w \quad \text{s.t.} \quad \mathbf{1}^\top w = 1$$

La solución analítica es:

$$w^* = \frac{\Sigma^{-1} \mathbf{1}}{\mathbf{1}^\top \Sigma^{-1} \mathbf{1}}$$

**Requisitos:**
1. Implementar `gmv_portfolio(Sigma)` usando la fórmula analítica con `np.linalg.solve`
2. Dividir los datos en train (80%) y test (20%) — **sin shuffle** (respeta el orden temporal)
3. Calcular $w^*_{\text{muestral}}$ y $w^*_{\text{PCA}}$ con $k=3$ sobre el conjunto train
4. Comparar **en el conjunto test**: retorno anualizado, volatilidad anualizada y Sharpe ratio
5. Graficar: (a) pesos de cada portafolio lado a lado, (b) retorno acumulado en test

In [ ]:
def gmv_portfolio(Sigma):
    """
    Portafolio de minima varianza global (GMV).

    w* = Sigma^{-1} @ 1 / (1.T @ Sigma^{-1} @ 1)

    Usa np.linalg.solve en lugar de invertir directamente.
    Pista: np.linalg.solve(Sigma, ones) equivale a Sigma^{-1} @ ones

    Parameters
    ----------
    Sigma : array (n, n) — matriz de covarianza

    Returns
    -------
    w : array (n,) — pesos del portafolio (suman 1)
    """
    # TODO: implementa la formula analitica
    pass


# Descomenta cuando hayas implementado gmv_portfolio y pca_shrinkage_cov

# Split temporal 80/20
# n_test  = int(len(ret_sec) * 0.20)
# ret_tr  = ret_sec.iloc[:-n_test].values
# ret_te  = ret_sec.iloc[-n_test:].values
# print(f'Train: {len(ret_tr)} dias | Test: {len(ret_te)} dias')

# Covarianzas estimadas en train
# Sigma_sample_tr = np.cov(ret_tr.T)
# Sigma_pca_tr, _, _ = pca_shrinkage_cov(ret_tr, k=3)

# Portafolios GMV
# w_sample = gmv_portfolio(Sigma_sample_tr)
# w_pca    = gmv_portfolio(Sigma_pca_tr)

# print(f'Pesos GMV muestral: min={w_sample.min():.3f}, max={w_sample.max():.3f}')
# print(f'Pesos GMV PCA:      min={w_pca.min():.3f},    max={w_pca.max():.3f}')

# TODO: calcula y reporta metricas OOT para ambos portafolios
# r_sample_oot = ret_te @ w_sample
# r_pca_oot    = ret_te @ w_pca

# def portfolio_metrics(r, freq=252):
#     return {
#         'Retorno anual': r.mean() * freq,
#         'Vol anual': r.std() * np.sqrt(freq),
#         'Sharpe': r.mean() / r.std() * np.sqrt(freq)
#     }

# TODO: grafico de pesos y retorno acumulado OOT
pass

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿Qué portafolio tiene mayor volatilidad OOT, el que usa $\hat{\Sigma}$ muestral o el que usa $\hat{\Sigma}_{\text{PCA}}$? ¿Por qué el GMV muestral no siempre produce el portafolio de menor varianza OOT?

> *Completa aquí*

**b)** Observa los pesos de cada portafolio. ¿Cuál tiene posiciones más extremas (muy largas o muy cortas)? ¿Cuál tiene posiciones más distribuidas entre sectores? Conecta esto con el número de condición de cada $\Sigma$.

> *Completa aquí*

---
## Parte 4 — Rolling PCA: Estabilidad y Detección de Regímenes *(25 pts)*

### Contexto

En la sesión vimos que ajustar PCA sobre toda la muestra asume estacionariedad — un supuesto cuestionable en mercados financieros. La varianza explicada por PC1 es un **indicador de correlación de mercado**: cuando sube, los activos se mueven juntos (régimen de crisis); cuando baja, los drivers sectoriales recuperan independencia.

El riesgo técnico al hacer rolling PCA es la **indeterminación de signo**: ventana a ventana, el algoritmo puede invertir el signo de un eigenvector arbitrariamente, produciendo series de loadings que oscilan sin razón económica.

### 4.1 — Rolling PCA con Alineación de Signos *(12 pts)*

Implementa `rolling_pca(X, window, k)` que retorna:
- `ev_pc1`: array con la varianza explicada por PC1 en cada ventana
- `loadings_pc1`: array con los loadings de PC1 en cada ventana

**Regla de alineación de signos:**
En cada ventana $t$, después de obtener los componentes, compara el componente $j$ con el de la ventana anterior:
$$\text{si} \quad \langle v_t^{(j)},\, v_{t-1}^{(j)} \rangle < 0 \quad \Rightarrow \quad v_t^{(j)} \leftarrow -v_t^{(j)}$$

**Para la primera ventana:** usa la convención de que la media de los loadings de PC1 sea positiva.

**Requisitos adicionales:**
1. Probar con ventanas $W \in \{60, 120, 252\}$ y comparar la suavidad de `ev_pc1`
2. Visualizar: (a) varianza PC1 en el tiempo para las 3 ventanas, (b) mapa de calor de loadings de PC1 (filas=activos, columnas=tiempo) para $W=120$
3. Identificar la fecha de **máxima correlación** (pico de `ev_pc1`) y verificar que coincide con un período de estrés conocido (COVID, corrección 2022)

In [ ]:
def rolling_pca(X, window, k, dates=None):
    """
    PCA rolling con alineacion de signos ventana a ventana.

    Parameters
    ----------
    X      : array (T, n)       — retornos
    window : int                — longitud de la ventana
    k      : int                — numero de componentes
    dates  : array-like or None — indices temporales para el output

    Returns
    -------
    ev_pc1      : array (T-window,) — varianza explicada PC1 en cada ventana
    loadings_t  : array (T-window, n) — loadings de PC1 en cada ventana
    out_dates   : array — fechas correspondientes (si se proveen)
    """
    T, n = X.shape
    ev_pc1     = []
    loadings_t = []
    prev_comp  = None

    for t in range(window, T + 1):
        # TODO paso 1: extraer ventana y ajustar PCA
        # window_data = X[t-window:t]
        # pca_t = PCA(n_components=k).fit(window_data)
        # comps = pca_t.components_.copy()   # (k, n)

        # TODO paso 2: alineacion de signos (primera ventana: media(comp[0]) > 0)
        # if prev_comp is None:
        #     for j in range(k):
        #         if comps[j].mean() < 0:
        #             comps[j] = -comps[j]
        # else:
        #     for j in range(k):
        #         if np.dot(comps[j], prev_comp[j]) < 0:
        #             comps[j] = -comps[j]

        # TODO paso 3: guardar varianza PC1 y loadings
        # ev_pc1.append(pca_t.explained_variance_ratio_[0])
        # loadings_t.append(comps[0].copy())
        # prev_comp = comps
        pass

    ev_pc1     = np.array(ev_pc1)
    loadings_t = np.array(loadings_t)
    out_dates  = np.array(dates[window:]) if dates is not None else None
    return ev_pc1, loadings_t, out_dates


# Descomenta cuando hayas implementado rolling_pca

# Usando los retornos de los 11 ETFs sectoriales
# ret_sec_arr = ret_sec.values
# dates_sec   = ret_sec.index

# windows = [60, 120, 252]
# results_rpca = {}
# for W in windows:
#     ev, load, d = rolling_pca(ret_sec_arr, W, k=3, dates=dates_sec)
#     results_rpca[W] = (ev, load, d)
#     max_ev_idx = np.argmax(ev)
#     print(f'W={W}: max ev_PC1={ev.max():.1%} en {d[max_ev_idx].date()}')

# TODO: Genera 2 paneles
# Panel izquierdo: ev_pc1 en el tiempo para las 3 ventanas
#   linea horizontal en el promedio de W=120
# Panel derecho: heatmap de loadings_t para W=120
#   (n filas=sectores, T-120 columnas=tiempo)
pass

### 4.2 — Varianza PC1 como Señal de Régimen *(13 pts)*

La varianza explicada por PC1 mide el **nivel de correlación sistémica** del mercado. Cuando supera un umbral $\theta$, el mercado está en un régimen de alta correlación (crisis). La estrategia es simple:

$$\text{exposición}_t = \begin{cases} 1 & \text{si } \text{ev}_{\text{PC1},t} \leq \theta \\ \alpha & \text{si } \text{ev}_{\text{PC1},t} > \theta \end{cases}$$

con $\alpha = 0.3$ (reduce la posición pero no sale completamente).

**Aplicación:** sobre el ETF SPY con $W=120$ y $\theta = \text{percentil}_{75}(\text{ev}_{\text{PC1}})$:

1. Calcular la señal `ev_PC1_t` usando `rolling_pca` sobre los 11 sectores
2. Construir la serie de exposición `exp_t` según la regla anterior (**lag 1 día para evitar leakage**)
3. Calcular retornos de la estrategia: $r_t^{\text{strat}} = \text{exp}_{t-1} \cdot r_t^{\text{SPY}}$
4. Comparar con buy-and-hold SPY en el período donde `ev_PC1` está disponible:
   - Retorno total, volatilidad anualizada, Sharpe ratio, Max Drawdown
5. Generar: (a) retorno acumulado comparado, (b) evolución de `ev_PC1` y exposición

In [ ]:
# Descomenta cuando hayas completado 4.1

# ev_120, _, dates_120 = results_rpca[120]
# theta = np.percentile(ev_120, 75)
# print(f'Umbral theta (p75): {theta:.3f}')

# Señal de exposicion (lag 1 dia)
# alpha_red = 0.3
# exposure  = np.where(ev_120 > theta, alpha_red, 1.0)
# exp_lagged = np.concatenate([[1.0], exposure[:-1]])  # lag 1 para evitar leakage

# Retornos SPY en el mismo periodo
# spy_all = yf.download('SPY', start='2020-01-01', end='2024-01-01',
#                       progress=False, auto_adjust=True)['Close']
# r_spy_all = spy_all.pct_change().dropna()
# r_spy_strat_period = r_spy_all.reindex(pd.DatetimeIndex(dates_120)).dropna()
# print(f'Dias estrategia: {len(r_spy_strat_period)}')

# TODO: asegurar que exp_lagged tiene el mismo largo que r_spy_strat_period
# r_strat = exp_lagged[:len(r_spy_strat_period)] * r_spy_strat_period.values
# r_bh    = r_spy_strat_period.values

# TODO: calcula y tabula metricas
# def metrics_portfolio(r, freq=252):
#     cum = np.cumprod(1 + r)
#     mdd = (cum / np.maximum.accumulate(cum) - 1).min()
#     return {
#         'Retorno Total':   cum[-1] - 1,
#         'Vol Anual':       r.std() * np.sqrt(freq),
#         'Sharpe':          r.mean() / r.std() * np.sqrt(freq),
#         'Max Drawdown':    mdd
#     }

# df_metrics_strat = pd.DataFrame({
#     'Buy & Hold SPY': metrics_portfolio(r_bh),
#     'Estrategia PC1': metrics_portfolio(r_strat),
# })
# display(df_metrics_strat.T.round(3))

# TODO: 2 paneles: retorno acumulado + ev_PC1 con exposicion
pass

**Responde (mínimo 2 oraciones por pregunta):**

**a)** ¿En qué fechas específicas fue mayor la varianza explicada por PC1? ¿Coinciden con eventos de mercado conocidos (COVID-19, subidas de tasas Fed 2022)?

> *Completa aquí*

**b)** Compara el Sharpe ratio y el Max Drawdown de la estrategia basada en PC1 vs buy-and-hold. ¿La reducción de exposición en regímenes de alta correlación mejora el perfil riesgo-retorno?

> *Completa aquí*

**c)** ¿Qué sesgo introduce el lag de 1 día en la señal de exposición? ¿Sería correcto usar `exposure` directamente sin lag? Argumenta en términos de leakage.

> *Completa aquí*

**d)** Compara las curvas de `ev_PC1` para $W=60$, $120$ y $252$. ¿Cuál ventana reacciona más rápido a los cambios de régimen? ¿Cuál tiene menor ruido? ¿Cómo elegirías $W$ en producción?

> *Completa aquí*

---
## Parte 5 — Sparse PCA y Robust PCA *(Bonus — 15 pts)*

### Contexto

Las extensiones de PCA presentadas al final de la sesión abordan dos limitaciones del PCA estándar:

1. **Sparse PCA:** produce loadings con muchos ceros → factores interpretables como "portfolios de pocos activos"
2. **Robust PCA:** separa la componente de bajo rango (factores comunes) de los outliers/jumps (componente sparse)

### 5.1 — Sparse PCA: Interpretabilidad vs Varianza Explicada *(5 pts)*

Aplica Sparse PCA con distintos valores de $\alpha \in \{0.1, 0.5, 1.0, 2.0\}$ sobre los retornos de los 11 ETFs.

Para cada $\alpha$, calcula:
- **Sparsidad:** `(components == 0).mean()` — fracción de loadings exactamente cero
- **Varianza explicada** (aproximada): `sum(loadings^2) / sum(Sigma_diag)` para PC1

Genera un gráfico de **sparsidad vs $\alpha$** y un heatmap comparativo de los loadings de PCA estándar vs Sparse PCA ($\alpha=0.5$).

In [ ]:
# TODO: Ajusta SparsePCA para alpha in [0.1, 0.5, 1.0, 2.0]
# alphas_spca = [0.1, 0.5, 1.0, 2.0]
# sparsity_list = []
# for alpha_s in alphas_spca:
#     spca_s = SparsePCA(n_components=3, alpha=alpha_s, random_state=42, max_iter=300)
#     spca_s.fit(ret_sec.values)
#     sparsity_list.append((spca_s.components_ == 0).mean())
#     print(f'alpha={alpha_s}: sparsidad={sparsity_list[-1]:.1%}')

# TODO: Genera 2 paneles
# Panel 1: sparsidad vs alpha
# Panel 2: heatmap comparativo PCA estandar vs SPCA (alpha=0.5)
#   Muestra los 3 primeros componentes de cada metodo
pass

**Responde:**

**a)** En el Sparse PCA con $\alpha=0.5$, ¿qué sectores tienen loadings cero en PC1? ¿Tiene sentido financiero que esos sectores sean excluidos del primer factor?

> *Completa aquí*

### 5.2 — Robust PCA: Separación de Factores y Jumps *(10 pts)*

Los retornos de activos contienen tanto factores comunes (movimientos sistemáticos de mercado) como saltos idiosincráticos (earnings surprise, M&A, eventos regulatorios). Robust PCA los separa:

$$X = L + S, \quad \min_{L,S} \|L\|_* + \lambda \|S\|_1$$

donde $L$ es la componente de bajo rango (factores) y $S$ es sparse (jumps).

Usa la implementación del algoritmo IALM (Inexact Augmented Lagrangian Method) de la clase:

**Requisitos:**
1. Copiar la función `robust_pca` de la clase e inyectar 20 jumps artificiales de magnitud $5\sigma$ en `ret_sec`
2. Aplicar Robust PCA sobre los datos con jumps
3. Verificar la recuperación: ¿cuántos de los 20 jumps inyectados detecta el componente $S$?
4. Comparar el PCA estándar sobre $X$ vs PCA sobre $L$ (la componente limpia): ¿cómo cambia la varianza explicada por PC1?
5. Generar el panel de 3 heatmaps: $X$ (con jumps) | $L$ (bajo rango) | $S$ (sparse)

In [ ]:
def robust_pca_ialm(M, lam=None, max_iter=500, tol=1e-7, mu=None):
    """
    Robust PCA via IALM (Candes et al. 2011).
    Descompone M = L + S donde L es low-rank y S es sparse.

    [Copiado de clase_05 — no modifiques esta funcion]
    """
    m, n = M.shape
    lam  = lam  or 1.0 / np.sqrt(max(m, n))
    mu   = mu   or m * n / (4 * np.abs(M).sum())
    mu_bar = mu * 1e7
    rho  = 1.5
    L, S, Y = np.zeros_like(M), np.zeros_like(M), np.zeros_like(M)
    norm_M   = np.linalg.norm(M, 'fro')

    def shrink(X, tau):
        return np.sign(X) * np.maximum(np.abs(X) - tau, 0)

    def svd_threshold(X, tau):
        U, s, Vt = np.linalg.svd(X, full_matrices=False)
        return U @ np.diag(np.maximum(s - tau, 0)) @ Vt

    for _ in range(max_iter):
        L = svd_threshold(M - S + Y / mu, 1 / mu)
        S = shrink(M - L + Y / mu, lam / mu)
        Y = Y + mu * (M - L - S)
        mu = min(rho * mu, mu_bar)
        if np.linalg.norm(M - L - S, 'fro') / norm_M < tol:
            break
    return L, S


# TODO 1: Inyecta 20 jumps artificiales de magnitud 5*sigma
# ret_noisy = ret_sec.values.copy()
# np.random.seed(42)
# jump_rows = np.random.choice(len(ret_noisy), 20, replace=False)
# jump_cols = np.random.choice(ret_noisy.shape[1], 20, replace=False)
# sigma_ret = ret_noisy.std(axis=0)
# for r, c in zip(jump_rows, jump_cols):
#     ret_noisy[r, c] += np.random.choice([-1, 1]) * 5 * sigma_ret[c]

# TODO 2: Aplica Robust PCA (puede tardar ~30 seg)
# L_rec, S_rec = robust_pca_ialm(ret_noisy, max_iter=300)

# TODO 3: Verifica recuperacion de jumps
# mask_S_detected = np.abs(S_rec) > 2 * sigma_ret
# detected = sum(mask_S_detected[r, c] for r, c in zip(jump_rows, jump_cols))
# print(f'Jumps inyectados: 20 | Detectados en S: {detected}')

# TODO 4: Compara varianza PC1 en X vs L
# pca_X = PCA(n_components=3).fit(ret_noisy)
# pca_L = PCA(n_components=3).fit(L_rec)
# print(f'Var PC1 en X (con jumps): {pca_X.explained_variance_ratio_[0]:.3f}')
# print(f'Var PC1 en L (low-rank):  {pca_L.explained_variance_ratio_[0]:.3f}')

# TODO 5: Panel de 3 heatmaps
# fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# Para cada matriz: ax.imshow(matriz.T, aspect='auto', cmap='RdBu_r', vmin=-vm, vmax=vm)
pass

**Responde:**

**a)** ¿Cuántos de los 20 jumps inyectados detectó el componente $S$? ¿En qué tipos de jumps falló la detección (magnitud baja, overlap con movimientos normales)?

> *Completa aquí*

**b)** ¿Cómo cambia la varianza explicada por PC1 entre PCA sobre $X$ (con jumps) vs PCA sobre $L$ (componente limpia)? ¿En qué dirección cambia y por qué?

> *Completa aquí*

**c)** En un pipeline de gestión de riesgo en producción, ¿antes o después de Robust PCA aplicarías el Rolling PCA de la Parte 4? Argumenta el orden correcto.

> *Completa aquí*

---
## Tabla de Puntajes

| Parte | Descripción | Pts |
|---|---|---|
| 1.1 | `pca_eigen` + `pca_svd` (error < 1e-10 vs sklearn) | 12 |
| 1.2 | `select_k_criteria` (var80 + Kaiser + Bai-Ng) + panel 4 gráficos × 2 datasets + 3 preguntas | 13 |
| 2.1 | PCA sobre $\Delta y$ + loadings vs NS teórico + correlaciones + gráfico | 10 |
| 2.2 | `dv01_factorial` + butterfly trade (sistema lineal) + P&L simulado + 3 preguntas | 15 |
| 3.1 | PCA $k=4$ + heatmap loadings + barplot PC1 + correlación SPY + 2 preguntas | 8 |
| 3.2 | `pca_shrinkage_cov` (verificación eigenvalores) + gráfico condicionamiento + heatmaps | 10 |
| 3.3 | `gmv_portfolio` + split temporal + métricas OOT + gráficos + 2 preguntas | 7 |
| 4.1 | `rolling_pca` con alineación de signos + 3 ventanas + heatmap loadings | 12 |
| 4.2 | Estrategia régimen PC1 + lag correcto + métricas + gráficos + 4 preguntas | 13 |
| **Total obligatorio** | | **100** |
| Bonus 5.1 | Sparse PCA × 4 alphas + sparsidad + heatmap comparativo + pregunta | 5 |
| Bonus 5.2 | Robust PCA + recuperación de jumps + comparación varianza + 3 preguntas | 10 |